# Data Analysis For All Weather Stations From IMS

In [5]:
import pandas as pd
from weather_engine.database import engine

CORE_FEATURES = pd.read_sql("SELECT * FROM clean_station_data LIMIT 0", engine).columns.tolist()
CORE_FEATURES = [c for c in CORE_FEATURES if c not in ('timestamp', 'station_id')]

missingness_cols_sql = ', '.join([
    f"ROUND(100.0 * SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS {col}_missingness"
    for col in CORE_FEATURES
])

query = f"""
    SELECT 
        station_id,
        COUNT(*) as total_rows,
        {missingness_cols_sql}
    FROM clean_station_data
    GROUP BY station_id
    ORDER BY total_rows ASC
"""

final_report_df = pd.read_sql(query, engine)

# Pull coordinates and join
coords_df = pd.read_sql("SELECT station_id, latitude, longitude FROM station_metadata", engine)
station_coords_df = final_report_df.merge(coords_df, on='station_id', how='left')

missingness_cols = [c for c in final_report_df.columns if c.endswith('_missingness')]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    styled_df = final_report_df.style.background_gradient(
        cmap='RdYlGn_r',
        subset=missingness_cols,
        vmin=0,
        vmax=100
    )
    display(styled_df)

,station_id,total_rows,rain_missingness,ws_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,u_vec_missingness,v_vec_missingness
0,502,12830,0.000000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000
1,500,15888,0.000000,5.560000,5.560000,0.030000,0.030000,0.030000,0.030000,5.560000,5.560000
2,411,28623,0.000000,8.310000,8.310000,6.560000,6.560000,6.560000,6.560000,8.310000,8.310000
3,2,52608,0.000000,0.020000,0.020000,0.040000,0.040000,0.040000,0.040000,0.020000,0.020000
4,6,52608,0.000000,0.450000,0.450000,0.450000,0.450000,0.450000,0.450000,0.450000,0.450000
5,8,52608,0.000000,3.390000,4.430000,0.010000,0.010000,0.010000,0.010000,4.260000,4.260000
6,10,52608,0.000000,0.020000,0.020000,0.010000,0.010000,0.010000,0.010000,0.020000,0.020000
7,11,52608,0.000000,0.690000,0.690000,0.690000,0.690000,0.690000,0.690000,0.690000,0.690000
8,13,52608,0.000000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
9,16,52608,0.000000,0.030000,0.030000,0.030000,0.030000,0.030000,0.030000,0.030000,0.030000


In [7]:
import pandas as pd
from weather_engine.database import engine

FEATURES = pd.read_sql("SELECT * FROM clean_station_data LIMIT 0", engine).columns.tolist()
FEATURES = [c for c in FEATURES if c not in ('timestamp', 'station_id')]

station_ids = pd.read_sql("SELECT DISTINCT station_id FROM clean_station_data ORDER BY station_id", engine)['station_id'].tolist()

records = []
for sid in station_ids:
    df = pd.read_sql(
        "SELECT timestamp, " + ", ".join(FEATURES) + " FROM clean_station_data WHERE station_id = :sid ORDER BY timestamp",
        engine, params={'sid': sid}
    )
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp').asfreq('1h')

    row = {'station_id': sid}
    row['data_from'] = df.index.min().strftime('%Y-%m-%d')
    row['data_to'] = df.index.max().strftime('%Y-%m-%d')

    best_gap_len = 0
    best_gap_start = None
    best_gap_end = None

    for feat in FEATURES:
        s = df[feat].isna()
        max_gap = 0
        if s.any():
            groups = (s != s.shift()).cumsum()
            for grp_id, grp in s.groupby(groups):
                if grp.iloc[0] and len(grp) > max_gap:
                    max_gap = len(grp)
                    if max_gap > best_gap_len:
                        best_gap_len = max_gap
                        best_gap_start = grp.index[0]
                        best_gap_end = grp.index[-1]
        row[f'{feat}_max_gap_h'] = max_gap

    row['worst_gap_start'] = best_gap_start.strftime('%Y-%m-%d') if best_gap_start else '-'
    row['worst_gap_end'] = best_gap_end.strftime('%Y-%m-%d') if best_gap_end else '-'
    records.append(row)

gap_df = pd.DataFrame(records).set_index('station_id')

meta_cols = ['data_from', 'data_to', 'worst_gap_start', 'worst_gap_end']
gap_cols = [c for c in gap_df.columns if c not in meta_cols]

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(gap_df.style.background_gradient(cmap='RdYlGn_r', subset=gap_cols, vmin=0, vmax=gap_df[gap_cols].max().max()))


,data_from,data_to,rain_max_gap_h,ws_max_gap_h,stdwd_max_gap_h,td_max_gap_h,rh_max_gap_h,tdmax_max_gap_h,tdmin_max_gap_h,u_vec_max_gap_h,v_vec_max_gap_h,worst_gap_start,worst_gap_end
station_id,,,,,,,,,,,,,
2,2019-12-31,2025-12-31,0,6,6,7,7,7,7,6,6,2020-08-03,2020-08-03
6,2019-12-31,2025-12-31,0,151,151,151,151,151,151,151,151,2020-07-01,2020-07-07
8,2019-12-31,2025-12-31,0,1225,1776,5,5,5,5,1682,1682,2020-06-28,2020-09-10
10,2019-12-31,2025-12-31,0,6,6,6,6,6,6,6,6,2024-09-05,2024-09-06
11,2019-12-31,2025-12-31,0,295,295,298,295,295,295,295,295,2024-11-08,2024-11-20
13,2019-12-31,2025-12-31,0,220,220,220,220,220,220,220,220,2020-11-21,2020-11-30
16,2019-12-31,2025-12-31,0,10,10,10,10,10,10,10,10,2020-12-14,2020-12-15
20,2019-12-31,2025-12-31,0,101,101,107,107,107,107,101,101,2021-12-31,2022-01-05
21,2019-12-31,2025-12-31,0,5,5,98,98,98,98,5,5,2020-06-21,2020-06-25
